# 01 - RFET Evidence and Classification

Synthetic engineering and validation evidence, not final regulatory capital.

Use this notebook with the [IMA package journey](../docs/PACKAGE_JOURNEY.md) and the executable [package demo](../examples/run_demo.py).

Raw inputs your upstream risk engine must emit: scenario P&L cubes with scenario metadata, risk-factor definitions, RFET real-price observations and evidence, stress-history series, PLA/backtesting observations, and NMRF valuation artifacts. The package dataset contract defines the committed fixture files, manifest lineage, and Arrow handoff: [`DATASET_CONTRACT.md`](../docs/DATASET_CONTRACT.md). The staged workflow is described in the [IMA package journey](../docs/PACKAGE_JOURNEY.md).

This notebook reviews Risk Factor Eligibility Test evidence for the committed `capital_run_v1` fixture. It uses the shared fixture loader in `examples.capital_run_fixture`; no independent observation data is created here.

Regulatory anchors: Basel MAR31 risk-factor modellability concepts; U.S. NPR 2.0 proposed-rule parameters for real-price observation thresholds and Type A / Type B NMRF classification cited below; EU CRR Article 325be as comparison context.

Prototype caution: the observation package is synthetic, deterministic fixture evidence. Classifications shown here are prototype model-validation evidence, not final regulatory capital determinations.


## Flow

```mermaid
flowchart LR
    A[RFET observations and evidence] --> B[Lineage and lookback filters]
    B --> C[Eligible observation counts]
    C --> D[Quantitative threshold checks]
    D --> E{Modellability status}
    E --> F[Modelled risk factors]
    E --> G[Type A and Type B NMRF routing]
```


## Public API happy path

This notebook loads RFET observations from the committed fixture and classifies modellability with the public RFET evidence workflow.


In [ ]:
from __future__ import annotations

from collections import Counter
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
for path in (ROOT, ROOT / "src"):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Markdown, display

from examples.capital_run_fixture import (
    load_capital_run_v1_fixture,
    policy_from_fixture,
    rfet_assessments_from_fixture,
)
from examples.notebook_utils import (
    IMA_CHART_COLORS,
    RFET_STATUS_PALETTE,
    apply_notebook_plot_style,
)
from frtb_ima import ModellabilityStatus

apply_notebook_plot_style()

fixture = load_capital_run_v1_fixture()
policy = policy_from_fixture(fixture)
assessments = rfet_assessments_from_fixture(fixture, policy)
expected_classifications = {
    name: ModellabilityStatus(status)
    for name, status in fixture.expected_outputs["classifications"].items()
}
ordered_names = tuple(risk_factor.name for risk_factor in fixture.risk_factors)
risk_factor_by_name = {risk_factor.name: risk_factor for risk_factor in fixture.risk_factors}

status_palette = RFET_STATUS_PALETTE


def markdown_table(headers: list[str], rows: list[list[str]]) -> Markdown:
    header = "| " + " | ".join(headers) + " |"
    separator = "| " + " | ".join(["---"] * len(headers)) + " |"
    body = ["| " + " | ".join(row) + " |" for row in rows]
    return Markdown("\n".join([header, separator, *body]))


def yes_no(value: bool) -> str:
    return "YES" if value else "NO"


display(
    Markdown(
        f"Loaded fixture `{fixture.params['run_id']}` for desk `{fixture.params['desk_id']}` "
        f"under `{policy.regime.value}`. RFET lookback: "
        f"`{policy.rfet_lookback_days}` days; short-LH threshold: "
        f"`{policy.rfet_short_lh_threshold}` observations; long-LH threshold: "
        f"`{policy.rfet_long_lh_threshold}` observations."
    )
)

## Implementation details / math verification - RFET thresholds and timelines

The remaining cells inspect observation counts, threshold checks, and evidence coverage over time.


## RFET Evidence Summary

The audit path applies source lineage, lookback-window, one-count-per-date, and bucket-representativeness checks before applying the quantitative threshold. The classification rule used here follows the prototype NPR 2.0 parameters cited above: qualitative pass plus quantitative pass is `MODELLABLE`; qualitative pass with quantitative fail is `TYPE_A_NMRF`; qualitative fail is `TYPE_B_NMRF`.

In [ ]:
rows: list[list[str]] = []
for name in ordered_names:
    assessment = assessments[name]
    risk_factor = risk_factor_by_name[name]
    expected_status = expected_classifications[name]
    rows.append(
        [
            name,
            risk_factor.risk_class.value,
            str(risk_factor.liquidity_horizon.value),
            str(assessment.eligible_observation_count),
            str(assessment.required_observations),
            yes_no(assessment.qualitative_pass),
            yes_no(assessment.quantitative_pass),
            assessment.modellability_status.value,
            "PASS" if assessment.modellability_status == expected_status else "CHECK",
        ]
    )

display(
    markdown_table(
        [
            "Risk factor",
            "Class",
            "LH days",
            "Eligible obs",
            "Required obs",
            "Qual pass",
            "Quant pass",
            "Classification",
            "Golden",
        ],
        rows,
    )
)

## Observation Counts Versus Thresholds

This chart is the main RFET sanity check: the bar is the de-duplicated eligible real-price count and the black marker is the policy threshold for that risk factor's liquidity horizon. Model validators should be able to see which classification is driven by sparse evidence versus qualitative failure.

In [ ]:
counts = np.asarray([assessments[name].eligible_observation_count for name in ordered_names])
required = np.asarray([assessments[name].required_observations for name in ordered_names])
colors = [status_palette[assessments[name].modellability_status.value] for name in ordered_names]
x = np.arange(len(ordered_names))

fig, ax = plt.subplots(figsize=(12, 5.2))
ax.bar(x, counts, color=colors, alpha=0.86, label="Eligible observations")
ax.scatter(x, required, color=IMA_CHART_COLORS["black"], marker="_", s=220, linewidths=2.2, label="Required")
for index, name in enumerate(ordered_names):
    status = assessments[name].modellability_status.value.replace("_", " ")
    ax.text(index, counts[index] + 0.65, status, ha="center", va="bottom", fontsize=8)
ax.set_xticks(x, ordered_names, rotation=45, ha="right")
ax.set_ylabel("Unique eligible observation dates")
ax.set_title("RFET eligible evidence against policy thresholds")
ax.legend(frameon=False, loc="upper right")
fig.tight_layout()

## Evidence Coverage Timeline

The timeline shows eligible observation dates inside the lookback window after de-duplication. The sparse NMRF evidence is intentionally visible rather than hidden in a scalar count.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5.6))
lookback_start = next(iter(assessments.values())).lookback_start
as_of_date = next(iter(assessments.values())).as_of_date
ax.axvspan(lookback_start, as_of_date, color=IMA_CHART_COLORS["light_gray"], alpha=0.55, label="Lookback window")
ax.axvline(as_of_date, color=IMA_CHART_COLORS["black"], linewidth=1.2, label="As-of date")

for y_value, name in enumerate(ordered_names):
    dates = list(assessments[name].eligible_observation_dates)
    if not dates:
        continue
    ax.scatter(
        dates,
        np.full(len(dates), y_value),
        s=22,
        color=status_palette[assessments[name].modellability_status.value],
        alpha=0.86,
    )

ax.set_yticks(np.arange(len(ordered_names)), ordered_names)
ax.invert_yaxis()
ax.set_xlabel("Observation date")
ax.set_title("Eligible RFET observation dates after one-count-per-date filtering")
ax.legend(frameon=False, loc="upper left")
fig.autofmt_xdate()
fig.tight_layout()

## Audit Exclusions and Status Mix

The fixture has no excluded RFET observations, but the notebook still surfaces the exclusion trail so future fixture revisions cannot silently hide date, source, or bucket-filtering effects.

In [ ]:
exclusion_counts = Counter(
    exclusion.reason.value
    for assessment in assessments.values()
    for exclusion in assessment.exclusions
)
exclusion_rows = (
    [[reason, str(count)] for reason, count in sorted(exclusion_counts.items())]
    if exclusion_counts
    else [["No exclusions", "0"]]
)
source_rows = [
    [
        name,
        str(assessments[name].source_count),
        yes_no(assessments[name].bucket_representative),
        yes_no(assessments[name].new_issuance_prorated),
    ]
    for name in ordered_names
]
display(markdown_table(["Exclusion reason", "Count"], exclusion_rows))
display(
    markdown_table(
        ["Risk factor", "Eligible sources", "Bucket representative", "New-issuance prorated"],
        source_rows,
    )
)

status_counts = Counter(assessments[name].modellability_status.value for name in ordered_names)
labels = list(status_palette)
values = np.asarray([status_counts.get(label, 0) for label in labels], dtype=float)
fig, ax = plt.subplots(figsize=(7.8, 4.4))
ax.bar(labels, values, color=[status_palette[label] for label in labels], alpha=0.88)
for index, value in enumerate(values):
    ax.text(index, value, f"{int(value)}", ha="center", va="bottom")
ax.set_ylabel("Risk factor count")
ax.set_title("RFET classification mix used by downstream IMCC/SES routing")
ax.set_xticks(np.arange(len(labels)), [label.replace("_", " ") for label in labels], rotation=15)
fig.tight_layout()

## See also

- [IMA package journey](../docs/PACKAGE_JOURNEY.md)
- [IMA dataset contract](../docs/DATASET_CONTRACT.md)
- [Client integration guide](../../../docs/CLIENT_INTEGRATION.md)
- [SBM package journey](../../frtb-sbm/docs/PACKAGE_JOURNEY.md)
- [DRC package journey](../../frtb-drc/docs/PACKAGE_JOURNEY.md)
- [RRAO package journey](../../frtb-rrao/docs/PACKAGE_JOURNEY.md)
- [CVA package journey](../../frtb-cva/docs/PACKAGE_JOURNEY.md)
